# AI4SAW — Notebook 01: Ingestion Demo

This notebook demonstrates the full ingestion pipeline:
1. Load a document (PDF, HTML, DOCX, or plaintext)
2. Chunk it with `RecursiveCharacterTextSplitter` (1000 token / 200 overlap)
3. Embed chunks via the configured provider
4. Store in ChromaDB with validated metadata
5. Verify retrieval

**Provider:** configure via `.env` — defaults to `PROVIDER=ollama` (free, offline).

In [ ]:
import sys
sys.path.insert(0, '..')

from ai4saw.core.config import settings
print(f'Provider: {settings.provider}')
print(f'Model:    {settings.default_model}')
print(f'Embedder: {settings.embedding_model}')
print(f'ChromaDB: {settings.chroma_persist_dir}')

## 1. Load a document

Replace `SOURCE` with any supported format: `.pdf`, `.html`, `.docx`, `.txt`, or a URL.

In [ ]:
from datetime import date
from ai4saw.ingestion.loaders import load_document

SOURCE = '../corpus/sample.txt'  # Replace with your document

docs = load_document(
    source=SOURCE,
    doc_type='report',
    language='en',
    geography='Bosnia',
    date_published=date(1996, 1, 1),
)

print(f'Loaded {len(docs)} page(s)')
print(f'\nFirst 500 chars of page 1:')
print(docs[0].page_content[:500])

## 2. Chunk

In [ ]:
from ai4saw.ingestion.chunker import chunk_documents, CHUNK_SIZE, CHUNK_OVERLAP

print(f'Chunk size: {CHUNK_SIZE} chars (~{CHUNK_SIZE//4} tokens)')
print(f'Overlap:    {CHUNK_OVERLAP} chars')

chunks = chunk_documents(docs)
print(f'\nProduced {len(chunks)} chunk(s)')

# Inspect first chunk metadata
print(f'\nChunk 0 metadata:')
for k, v in chunks[0].metadata.items():
    print(f'  {k}: {v}')

## 3. Embed and store

In [ ]:
from ai4saw.ingestion.embedder import embed_and_store

store = embed_and_store(chunks)
print(f'ChromaDB collection: {settings.chroma_collection}')
print(f'Total chunks in store: {store._collection.count()}')

## 4. Verify retrieval

In [ ]:
query = 'forced labour detention'

results = store.similarity_search_with_relevance_scores(query, k=3)

print(f'Top 3 results for: {query!r}\n')
for i, (doc, score) in enumerate(results, 1):
    print(f'--- Result {i} (score={score:.4f}) ---')
    print(f'Source: {doc.metadata.get("source_filename")}')
    print(doc.page_content[:300])
    print()

## 5. Corpus ingest (batch)

To ingest an entire directory of documents, use `load_corpus` and pipe into the chunker and embedder.

In [ ]:
from pathlib import Path
from ai4saw.ingestion.loaders import load_corpus

corpus_dir = Path('../corpus')

if corpus_dir.exists() and any(corpus_dir.iterdir()):
    all_docs = load_corpus(corpus_dir, doc_type='report', language='en', geography='Bosnia')
    all_chunks = chunk_documents(all_docs)
    embed_and_store(all_chunks)
    print(f'Total chunks now in store: {store._collection.count()}')
else:
    print('No files in corpus/ directory — add documents and re-run.')